In [ ]:
import pandas as pd
import numpy as np

In [ ]:
discovery_set = pd.read_csv("/Users/ashfrana/Desktop/code/abcd_sex_pfn_replication/discovery and replication sample setup scripts/data/discovery_sample_siblings_removed_071524.csv")
replication_set = pd.read_csv("/Users/ashfrana/Desktop/code/abcd_sex_pfn_replication/discovery and replication sample setup scripts/data/replication_sample_siblings_removed_071524.csv")

In [ ]:
import pyreadr

In [ ]:
result = pyreadr.read_r("/Users/ashfrana/Desktop/code/abcd_sex_pfn_replication/demographics_table/data/DEAP-siteID.rds")
dem_data = result[None]
dem_data = dem_data[dem_data['event_name'] == 'baseline_year_1_arm_1']
dem_data = dem_data[['src_subject_id', 'household.income', 'race.6level']]
new_subjectkey = []
for key in dem_data['src_subject_id'].values:
    new_subjectkey.append(key[5:])
dem_data['subjectkey'] = new_subjectkey
dem_data = dem_data[['subjectkey', 'household.income', 'race.6level']]
dem_data

In [ ]:
hormone_hse_ert_data = pd.read_csv("/Users/ashfrana/Desktop/code/abcd_sex_pfn_replication/univariate_analysis/pubertal_analyses/puberty_data_local/ph_y_sal_horm.csv")
hormone_data = hormone_hse_ert_data[hormone_hse_ert_data['eventname'] == 'baseline_year_1_arm_1']
subjects_hormones = []
for key in hormone_data['src_subject_id'].values:
    subjects_hormones.append(key[5:])
hormone_data['subjectkey'] = subjects_hormones
hormone_data = hormone_data[['subjectkey', 'hormone_scr_ert_mean', 'hormone_scr_hse_mean']]
hormone_data

In [ ]:
dhea_data = pd.read_csv("/Users/ashfrana/Desktop/code/abcd_sex_pfn_replication/univariate_analysis/pubertal_analyses/puberty_data_local/abcd_hsss01.txt", sep='\t')
dhea_data = dhea_data.drop([0], axis=0)
dhea_data = dhea_data[dhea_data['eventname'] == 'baseline_year_1_arm_1']
subjects_dhea = []
for key in dhea_data['src_subject_id'].values:
    subjects_dhea.append(key[5:])
dhea_data['subjectkey'] = subjects_dhea
dhea_data = dhea_data[['subjectkey', 'hormone_scr_dhea_mean']]
dhea_data

In [ ]:
all_hormones_data = dhea_data.merge(hormone_data, on='subjectkey', how='left')
all_hormones_data

In [ ]:
pds_data = pd.read_csv("/Users/ashfrana/Desktop/code/abcd_sex_pfn_replication/univariate_analysis/pubertal_analyses/puberty_data_local/ph_p_pds.csv")
pds_data = pds_data[pds_data['eventname'] == 'baseline_year_1_arm_1']
subjects_pds = []
for key in pds_data['src_subject_id'].values:
    subjects_pds.append(key[5:])
pds_data['subjectkey'] = subjects_pds
pds_data = pds_data[['subjectkey', 'pds_p_ss_female_category', 'pds_p_ss_male_category']]
pds_data

In [ ]:
disc_w_race = discovery_set.merge(dem_data[['subjectkey', 'race.6level']], on='subjectkey', how='left')
disc_w_race

In [ ]:
disc_w_hormones = discovery_set.merge(all_hormones_data, on='subjectkey', how='left')
disc_w_hormones

In [ ]:
rep_w_race = replication_set.merge(dem_data[['subjectkey', 'race.6level']], on='subjectkey', how='left')
rep_w_race

In [ ]:
income_data = dem_data[['subjectkey','household.income']].drop_duplicates()
income_data

In [ ]:
discovery_dem = disc_w_race.merge(income_data, on='subjectkey', how='left')
# Keep values if filled, otherwise set to NaN
discovery_dem = discovery_dem.merge(all_hormones_data, on='subjectkey', how='left')
discovery_dem = discovery_dem.merge(pds_data, on='subjectkey', how='left')
discovery_dem

In [ ]:
replication_dem = rep_w_race.merge(income_data, on='subjectkey', how='left')
replication_dem = replication_dem.merge(all_hormones_data, on='subjectkey', how='left')
replication_dem = replication_dem.merge(pds_data, on='subjectkey', how='left')
replication_dem

In [ ]:
total_dem = pd.concat([discovery_dem, replication_dem], axis=0)
total_dem

In [ ]:
def get_means_sd(dem_data, variable):
    if variable == "interview_age":
        dem_data = dem_data[variable].values/12
    else:
        dem_data = dem_data[variable].values
        dem_data = [float(val) for val in dem_data]

    #print(dem_data)
    mean = np.mean(dem_data)
    #print(mean)
    sd = np.std(dem_data)
    mean_sd = f"{round(mean, 2)} ({round(sd, 2)})"
    return mean_sd

In [ ]:
sex_counts = discovery_dem['sex'].value_counts()
race_counts = discovery_dem['race.6level'].value_counts()
income_counts = discovery_dem['household.income'].value_counts()

race_na = discovery_dem['race.6level'].value_counts(dropna=False)
income_na = discovery_dem['household.income'].value_counts(dropna=False)

disc_dem_df = pd.DataFrame()

disc_dem_df['age'] = [get_means_sd(discovery_dem, 'interview_age')]
disc_dem_df[['M', 'F']] = [sex_counts[['M', 'F']].values]
disc_dem_df[['White', 'Black', 'Asian', 'AIAN/NHPI', 'Mixed', 'Other']] = [race_counts[['White', 'Black', 'Asian', 'AIAN/NHPI', 'Mixed', 'Other']].values]
disc_dem_df['race_NA'] = [race_na[np.nan]]
disc_dem_df[['[<50K]', '[>=50K & <100K]', '[>=100K]']] = [income_counts[['[<50K]', '[>=50K & <100K]', '[>=100K]']].values]
disc_dem_df['income_NA'] = [income_na[np.nan]]

dhea_no_na = discovery_dem[discovery_dem['hormone_scr_dhea_mean'].isna() == False]
disc_dem_df['DHEA'] = get_means_sd(dhea_no_na, 'hormone_scr_dhea_mean')
#disc_dem_df['hormone_scr_dhea_mean_NA'] = len(discovery_dem[discovery_dem['hormone_scr_dhea_mean'].isna()])

ert_no_na = discovery_dem[discovery_dem['hormone_scr_ert_mean'].isna() == False]
disc_dem_df['Testosterone'] = get_means_sd(ert_no_na, 'hormone_scr_ert_mean')
#disc_dem_df['hormone_scr_ert_mean_NA'] = len(discovery_dem[discovery_dem['hormone_scr_ert_mean'].isna()])

hse_no_na = discovery_dem[discovery_dem['hormone_scr_hse_mean'].isna() == False]
disc_dem_df['Estradiol'] = get_means_sd(hse_no_na, 'hormone_scr_hse_mean')
#disc_dem_df['hormone_scr_hse_mean_NA'] = len(discovery_dem[discovery_dem['hormone_scr_hse_mean'].isna()])

#disc_pds_female_no_na = discovery_dem[discovery_dem['pds_p_ss_female_category'].isna() == False]
#disc_pds_female_category = f"{get_means_sd(disc_pds_female_no_na, 'pds_p_ss_female_category')} (n={len(disc_pds_female_no_na)})"
disc_pds_female_counts = discovery_dem['pds_p_ss_female_category'].value_counts()
disc_dem_df[['1f', '2f', '3f', '4f', '5f']] = disc_pds_female_counts[[1,2,3,4,5]].values

#disc_pds_male_no_na = discovery_dem[discovery_dem['pds_p_ss_male_category'].isna() == False]
#disc_pds_male_category = f"{get_means_sd(disc_pds_male_no_na, 'pds_p_ss_male_category')} (n={len(disc_pds_male_no_na)})"
disc_pds_male_counts = discovery_dem['pds_p_ss_male_category'].value_counts()
disc_dem_df[['1m', '2m', '3m', '4m']] = disc_pds_male_counts[[1,2,3,4]].values

# print(sex_counts)
# print(race_counts)
# print(income_counts)
print(disc_dem_df.T)
#print(sex_counts[sex_counts.index].values)

In [ ]:
rep_sex_counts = replication_dem['sex'].value_counts()
rep_race_counts = replication_dem['race.6level'].value_counts()
rep_income_counts = replication_dem['household.income'].value_counts()

rep_race_na = replication_dem['race.6level'].value_counts(dropna=False)
rep_income_na = replication_dem['household.income'].value_counts(dropna=False)

rep_dem_df = pd.DataFrame()

rep_dem_df['age'] = [get_means_sd(replication_dem, 'interview_age')]
rep_dem_df[['M', 'F']] = [rep_sex_counts[['M', 'F']].values]
rep_dem_df[['White', 'Black', 'Asian', 'AIAN/NHPI', 'Mixed', 'Other']] = [rep_race_counts[['White', 'Black', 'Asian', 'AIAN/NHPI', 'Mixed', 'Other']].values]
rep_dem_df['race_NA'] = [rep_race_na[np.nan]]
rep_dem_df[['[<50K]', '[>=50K & <100K]', '[>=100K]']] = [rep_income_counts[['[<50K]', '[>=50K & <100K]', '[>=100K]']].values]
rep_dem_df['income_NA'] = [rep_income_na[np.nan]]

dhea_no_na_rep = replication_dem[replication_dem['hormone_scr_dhea_mean'].isna() == False]
rep_dem_df['DHEA'] = get_means_sd(dhea_no_na_rep, 'hormone_scr_dhea_mean')
#rep_dem_df['hormone_scr_dhea_mean_NA'] = len(replication_dem[replication_dem['hormone_scr_dhea_mean'].isna()])

ert_no_na_rep = replication_dem[replication_dem['hormone_scr_ert_mean'].isna() == False]
rep_dem_df['Testosterone'] = get_means_sd(ert_no_na_rep, 'hormone_scr_ert_mean')
#rep_dem_df['hormone_scr_ert_mean_NA'] = len(replication_dem[replication_dem['hormone_scr_ert_mean'].isna()])

hse_no_na_rep = replication_dem[replication_dem['hormone_scr_hse_mean'].isna() == False]
rep_dem_df['Estradiol'] = get_means_sd(hse_no_na_rep, 'hormone_scr_hse_mean')
#rep_dem_df['hormone_scr_hse_mean_NA'] = len(replication_dem[replication_dem['hormone_scr_hse_mean'].isna()])


#rep_pds_female_no_na = replication_dem[discovery_dem['pds_p_ss_female_category'].isna() == False]
#rep_pds_female_category = f"{get_means_sd(rep_pds_female_no_na, 'pds_p_ss_female_category')} (n={len(disc_pds_female_no_na)})"
rep_pds_female_counts = replication_dem['pds_p_ss_female_category'].value_counts()
rep_dem_df[['1f', '2f', '3f', '4f', '5f']] = rep_pds_female_counts[[1,2,3,4,5]].values

#rep_pds_male_no_na = replication_dem[discovery_dem['pds_p_ss_male_category'].isna() == False]
#rep_pds_male_category = f"{get_means_sd(rep_pds_male_no_na, 'pds_p_ss_male_category')} (n={len(disc_pds_male_no_na)})"
rep_pds_male_counts = replication_dem['pds_p_ss_male_category'].value_counts()
rep_dem_df[['1m', '2m', '3m', '4m']] = rep_pds_male_counts[[1,2,3,4]].values

print(rep_sex_counts)
print(rep_race_counts)
print(rep_income_counts)
print(rep_dem_df.T)

In [ ]:
total_sex_counts = total_dem['sex'].value_counts()
total_race_counts = total_dem['race.6level'].value_counts()
total_income_counts = total_dem['household.income'].value_counts()

total_race_na = total_dem['race.6level'].value_counts(dropna=False)
total_income_na = total_dem['household.income'].value_counts(dropna=False)

total_dem_df = pd.DataFrame()
total_dem_df['age'] = [get_means_sd(total_dem, 'interview_age')]
total_dem_df[['M', 'F']] = [total_sex_counts[['M', 'F']].values]
total_dem_df[['White', 'Black', 'Asian', 'AIAN/NHPI', 'Mixed', 'Other']] = [total_race_counts[['White', 'Black', 'Asian', 'AIAN/NHPI', 'Mixed', 'Other']].values]
total_dem_df['race_NA'] = [total_race_na[np.nan]]
total_dem_df[['[<50K]', '[>=50K & <100K]', '[>=100K]']] = [total_income_counts[['[<50K]', '[>=50K & <100K]', '[>=100K]']].values]
total_dem_df['income_NA'] = [total_income_na[np.nan]]


dhea_no_na_total = total_dem[total_dem['hormone_scr_dhea_mean'].isna() == False]
total_dem_df['DHEA'] = get_means_sd(dhea_no_na_total, 'hormone_scr_dhea_mean')
#total_dem_df['hormone_scr_dhea_mean_NA'] = len(total_dem[total_dem['hormone_scr_dhea_mean'].isna()])

ert_no_na_total = total_dem[total_dem['hormone_scr_ert_mean'].isna() == False]
total_dem_df['Testosterone'] = get_means_sd(ert_no_na_total, 'hormone_scr_ert_mean')
#total_dem_df['hormone_scr_ert_mean_NA'] = len(total_dem[total_dem['hormone_scr_ert_mean'].isna()])

hse_no_na_total = total_dem[total_dem['hormone_scr_hse_mean'].isna() == False]
total_dem_df['Estradiol'] = get_means_sd(hse_no_na_total, 'hormone_scr_hse_mean')
#total_dem_df['hormone_scr_hse_mean_NA'] = len(total_dem[total_dem['hormone_scr_hse_mean'].isna()])


# total_pds_female_no_na = total_dem[total_dem['pds_p_ss_female_category'].isna() == False]
# total_dem_df[f'PDS Female'] = f"{get_means_sd(total_pds_female_no_na, 'pds_p_ss_female_category')} (n={len(total_pds_female_no_na)})"
# #total_dem_df['pds_female_NA'] = [total_pds_female_category[np.nan]]

# total_pds_male_no_na = total_dem[total_dem['pds_p_ss_male_category'].isna() == False]
# total_dem_df[f'PDS Male'] = f"{get_means_sd(total_pds_male_no_na, 'pds_p_ss_male_category')} (n={len(total_pds_male_no_na)})"
# #total_dem_df['pds_male_NA'] = [total_pds_male_category[np.nan]]

total_pds_female_counts = total_dem['pds_p_ss_female_category'].value_counts()
total_dem_df[['1f', '2f', '3f', '4f', '5f']] = total_pds_female_counts[[1,2,3,4,5]].values

total_pds_male_counts = total_dem['pds_p_ss_male_category'].value_counts()
total_dem_df[['1m', '2m', '3m', '4m']] = total_pds_male_counts[[1,2,3,4]].values


print(total_sex_counts)
print(total_race_counts)
print(total_income_counts)
print(total_dem_df.T)
print(f"PDS Female n={len(total_dem[total_dem['pds_p_ss_female_category'].isna() == False])}, PDS Male n={len(total_dem[total_dem['pds_p_ss_male_category'].isna() == False])}")

In [ ]:
def create_percentages(demographic_df, type):
    if type == "disc":
        total = 3240
    elif type == "rep":
        total = 3197
    else:
        total = 6437

    # pds female
    pds_female_total = demographic_df[['1f', '2f', '3f', '4f', '5f']].values.sum()

    # pds male
    pds_male_total = demographic_df[['1m', '2m', '3m', '4m',]].values.sum()

    for col in demographic_df:
        if col not in ["age", "DHEA", "Testosterone", 'Estradiol', '1m', '2m', '3m', '4m', '1f', '2f', '3f', '4f', '5f']:
            #print(col)
            total = total
            value = demographic_df[col].values[0]
            print(value)
            percentage = round((value / total)*100, 2)
            new_value = f"{value} ({percentage}%)"
            demographic_df[col] = new_value
        
        elif "m" in col and col not in ["age", "DHEA", "Testosterone", 'Estradiol']:
            total = pds_male_total
            value = demographic_df[col].values[0]
            print(value)
            percentage = round((value / total)*100, 2)
            new_value = f"{value} ({percentage}%)"
            demographic_df[col] = new_value
        elif "f" in col and col not in ["age", "DHEA", "Testosterone", 'Estradiol']:
            total = pds_female_total
            value = demographic_df[col].values[0]
            print(value)
            percentage = round((value / total)*100, 2)
            new_value = f"{value} ({percentage}%)"
            demographic_df[col] = new_value
            
        
    final_df = demographic_df.T
    return final_df

final_discovery_df = create_percentages(disc_dem_df, "disc")
final_discovery_df

In [ ]:
final_discovery_df.rename(index=str, columns={0: "Discovery"}, inplace=True)
final_discovery_df

In [ ]:
final_replication_df = create_percentages(rep_dem_df, "rep")
final_replication_df

In [ ]:
final_replication_df.rename(index=str, columns={0: "Replication"}, inplace=True)
final_replication_df

In [ ]:
final_total_df = create_percentages(total_dem_df, "total")
final_total_df

In [ ]:
final_total_df.rename(index=str, columns={0: "Total"}, inplace=True)
final_total_df

In [ ]:
final_df = pd.concat([final_discovery_df, final_replication_df, final_total_df], axis=1)
final_df

In [ ]:
final_df.to_csv("/Users/ashfrana/Desktop/code/abcd_sex_pfn_replication/demographics_table/ABCD_pfns_demographic_table.csv")